# Ch.2 — Efficient Architectures (MobileNet, EfficientNet)

**The breakthrough:** Depthwise separable convolutions reduce compute by 8× with minimal accuracy loss.

**What you'll build:** MobileNetV2 from scratch, compare with ResNet-50, deploy on edge device simulation.

**Dataset:** CIFAR-10 (proxy for SmallVal AI retail shelf dataset).

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
import time

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Dark theme for plots
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#1a1a2e'

## 2. Load CIFAR-10 Dataset

In [ ]:
# Same data loading as Ch.1
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')
print(f'Training samples: {len(trainset)}, Test samples: {len(testset)}')

## 3. Build MobileNetV2 from Scratch

**Key components:**
- Depthwise Separable Conv: Depthwise 3×3 + Pointwise 1×1
- Inverted Residual Block: Expand → Depthwise → Project (linear) + skip

In [ ]:
class InvertedResidual(nn.Module):
    """MobileNetV2 Inverted Residual Block"""
    def __init__(self, in_channels, out_channels, stride, expansion_ratio=6):
        super().__init__()
        self.stride = stride
        self.use_residual = (stride == 1 and in_channels == out_channels)
        
        hidden_dim = in_channels * expansion_ratio
        
        layers = []
        # Expansion (skip if expansion_ratio == 1)
        if expansion_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, kernel_size=1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True)
            ])
        
        # Depthwise
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, kernel_size=3, stride=stride,
                     padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True)
        ])
        
        # Projection (LINEAR — no ReLU!)
        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels)
        ])
        
        self.conv = nn.Sequential(*layers)
    
    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        else:
            return self.conv(x)


class MobileNetV2(nn.Module):
    """MobileNetV2 architecture (adapted for CIFAR-10)"""
    def __init__(self, num_classes=10, width_mult=1.0):
        super().__init__()
        
        # Initial conv (adapted for 32×32 input)
        input_channels = int(32 * width_mult)
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, input_channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(input_channels),
            nn.ReLU6(inplace=True)
        )
        
        # Inverted residual blocks [t, c, n, s]
        block_config = [
            [1, 16, 1, 1],
            [6, 24, 2, 1],  # stride=1 for CIFAR-10
            [6, 32, 3, 2],
            [6, 64, 4, 2],
            [6, 96, 3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]
        
        layers = []
        for t, c, n, s in block_config:
            out_channels = int(c * width_mult)
            for i in range(n):
                stride = s if i == 0 else 1
                layers.append(InvertedResidual(input_channels, out_channels, stride, t))
                input_channels = out_channels
        
        self.blocks = nn.Sequential(*layers)
        
        # Final conv
        last_channels = int(1280 * width_mult) if width_mult > 1.0 else 1280
        self.conv2 = nn.Sequential(
            nn.Conv2d(input_channels, last_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(last_channels),
            nn.ReLU6(inplace=True)
        )
        
        # Classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(last_channels, num_classes)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.blocks(x)
        x = self.conv2(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# Instantiate models with different width multipliers
model_full = MobileNetV2(num_classes=10, width_mult=1.0).to(device)
model_compact = MobileNetV2(num_classes=10, width_mult=0.75).to(device)

print(f'MobileNetV2 (α=1.0):  {sum(p.numel() for p in model_full.parameters()):,} parameters')
print(f'MobileNetV2 (α=0.75): {sum(p.numel() for p in model_compact.parameters()):,} parameters')

## 4. FLOPs Analysis: MobileNetV2 vs Standard Conv

Calculate computational cost for one layer.

In [ ]:
# Example: 56×56×128 → 56×56×256 feature map
H, W = 56, 56
C_in, C_out = 128, 256
k = 3

# Standard 3×3 conv
standard_flops = k**2 * C_in * C_out * H * W
standard_params = k**2 * C_in * C_out

# Depthwise separable (depthwise + pointwise)
depthwise_flops = k**2 * C_in * H * W
pointwise_flops = C_in * C_out * H * W
depsep_flops = depthwise_flops + pointwise_flops

depthwise_params = k**2 * C_in
pointwise_params = C_in * C_out
depsep_params = depthwise_params + pointwise_params

# Speedup
flops_speedup = standard_flops / depsep_flops
params_speedup = standard_params / depsep_params

print('Computational Cost Comparison (56×56×128 → 56×56×256):\n')
print(f'Standard 3×3 Conv:')
print(f'  FLOPs:  {standard_flops:,}')
print(f'  Params: {standard_params:,}\n')

print(f'Depthwise Separable Conv:')
print(f'  Depthwise FLOPs:  {depthwise_flops:,}')
print(f'  Pointwise FLOPs:  {pointwise_flops:,}')
print(f'  Total FLOPs:      {depsep_flops:,}')
print(f'  Total Params:     {depsep_params:,}\n')

print(f'Speedup:')
print(f'  FLOPs:  {flops_speedup:.2f}×')
print(f'  Params: {params_speedup:.2f}×')

# Theoretical speedup formula
theoretical_speedup = (k**2 * C_out) / (k**2 + C_out)
print(f'  Theoretical (k²·C_out / (k²+C_out)): {theoretical_speedup:.2f}×')

## 5. Training Setup

In [ ]:
# Train MobileNetV2 (α=1.0)
model = model_full
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=4e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print('Training MobileNetV2 (α=1.0)...')

## 6. Train MobileNetV2

Train for 20 epochs (same as ResNet-18 in Ch.1 for fair comparison).

In [ ]:
def train_epoch(model, trainloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in tqdm(trainloader, desc='Training', leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(trainloader), 100. * correct / total


def test_epoch(model, testloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in tqdm(testloader, desc='Testing', leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(testloader), 100. * correct / total


# Training loop
num_epochs = 20
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, trainloader, criterion, optimizer, device)
    test_loss, test_acc = test_epoch(model, testloader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    scheduler.step()
    
    print(f'Epoch {epoch+1}/{num_epochs}: '
          f'Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}% | '
          f'Test Loss={test_loss:.4f}, Test Acc={test_acc:.2f}%')

print(f'\nFinal Test Accuracy: {test_accs[-1]:.2f}%')

## 7. Inference Latency Benchmark

Compare inference speed: MobileNetV2 vs ResNet-18 (from Ch.1).

In [ ]:
# Benchmark function
def benchmark_inference(model, input_shape=(1, 3, 32, 32), num_runs=100):
    model.eval()
    dummy_input = torch.randn(input_shape).to(device)
    
    # Warmup
    for _ in range(10):
        _ = model(dummy_input)
    
    # Benchmark
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    start = time.time()
    for _ in range(num_runs):
        _ = model(dummy_input)
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    end = time.time()
    
    avg_time = (end - start) / num_runs * 1000  # Convert to ms
    return avg_time


# Compare MobileNetV2 variants
latency_full = benchmark_inference(model_full)
latency_compact = benchmark_inference(model_compact)

print('Inference Latency Comparison (CIFAR-10 32×32 input):\n')
print(f'MobileNetV2 (α=1.0):  {latency_full:.2f} ms')
print(f'MobileNetV2 (α=0.75): {latency_compact:.2f} ms')
print(f'Speedup (α=0.75):     {latency_full / latency_compact:.2f}×')

# Note: For real edge device (Jetson Nano), latency would be ~35ms for 224×224 input

## 8. Visualize Efficiency Tradeoff

Plot accuracy vs model size/FLOPs for different architectures.

In [ ]:
# Simulated data for different architectures (ImageNet top-1 accuracy)
architectures = [
    ('ResNet-18', 11.7, 1.8, 69.8),
    ('ResNet-50', 25.6, 4.1, 76.1),
    ('MobileNetV2 (α=0.5)', 2.0, 0.1, 65.4),
    ('MobileNetV2 (α=0.75)', 2.6, 0.2, 69.8),
    ('MobileNetV2 (α=1.0)', 3.5, 0.3, 72.0),
    ('EfficientNet-B0', 5.3, 0.39, 77.3),
    ('EfficientNet-B3', 12.0, 1.8, 81.7),
]

names, params, gflops, accuracies = zip(*architectures)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Accuracy vs Parameters
colors = ['#ef4444', '#ef4444', '#3b82f6', '#3b82f6', '#3b82f6', '#10b981', '#10b981']
ax1.scatter(params, accuracies, c=colors, s=200, alpha=0.7, edgecolors='white', linewidth=2)
for i, name in enumerate(names):
    ax1.annotate(name, (params[i], accuracies[i]), 
                 textcoords='offset points', xytext=(0, 10), ha='center',
                 fontsize=9, color='white', fontweight='bold')
ax1.set_xlabel('Parameters (millions)', fontsize=12, fontweight='bold')
ax1.set_ylabel('ImageNet Top-1 Accuracy (%)', fontsize=12, fontweight='bold')
ax1.set_title('Accuracy vs Model Size', fontsize=14, fontweight='bold')
ax1.grid(alpha=0.3)
ax1.axhline(y=70, color='#f59e0b', linestyle='--', alpha=0.5, label='70% threshold')
ax1.legend(fontsize=10)

# Plot 2: Accuracy vs GFLOPs
ax2.scatter(gflops, accuracies, c=colors, s=200, alpha=0.7, edgecolors='white', linewidth=2)
for i, name in enumerate(names):
    ax2.annotate(name, (gflops[i], accuracies[i]), 
                 textcoords='offset points', xytext=(0, 10), ha='center',
                 fontsize=9, color='white', fontweight='bold')
ax2.set_xlabel('FLOPs (GFLOPs)', fontsize=12, fontweight='bold')
ax2.set_ylabel('ImageNet Top-1 Accuracy (%)', fontsize=12, fontweight='bold')
ax2.set_title('Accuracy vs Computational Cost', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)
ax2.axhline(y=70, color='#f59e0b', linestyle='--', alpha=0.5, label='70% threshold')
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('img/ch02-efficiency-tradeoff.png', dpi=150, facecolor='#1a1a2e')
plt.show()

print('📊 Key insight: MobileNetV2 and EfficientNet achieve ResNet-level accuracy with 5–8× fewer parameters!')

## 9. Ablation: Linear Bottleneck Importance

Compare inverted residual with ReLU vs without ReLU after final projection.

In [ ]:
class InvertedResidualWithReLU(nn.Module):
    """WRONG version with ReLU after projection (for comparison)"""
    def __init__(self, in_channels, out_channels, stride, expansion_ratio=6):
        super().__init__()
        self.stride = stride
        self.use_residual = (stride == 1 and in_channels == out_channels)
        
        hidden_dim = in_channels * expansion_ratio
        
        layers = []
        if expansion_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.ReLU6(inplace=True)
            ])
        
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride=stride, padding=1, groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.ReLU6(inplace=True)
        ])
        
        # WRONG: ReLU after projection!
        layers.extend([
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU6(inplace=True)  # ❌ BAD!
        ])
        
        self.conv = nn.Sequential(*layers)
    
    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        else:
            return self.conv(x)


print('🔍 Ablation: Linear Bottleneck vs ReLU After Projection\n')
print('Standard MobileNetV2 (linear bottleneck):')
print('  Expected accuracy: 91–92% on CIFAR-10')
print('\nMobileNetV2 with ReLU after projection (wrong):')
print('  Expected accuracy: 88–89% on CIFAR-10 (-3% loss!)')
print('\n💡 Why: ReLU destroys information in low-dimensional spaces (manifold hypothesis)')

## 10. What Can Go Wrong — Demonstration

In [ ]:
print('🔍 Common MobileNet pitfalls:\n')

print('❌ TRAP 1: ReLU after final projection (linear bottleneck)')
print('   Wrong:  project → BN → ReLU6')
print('   Right:  project → BN (no activation)')
print('   Impact: -2 to -3% accuracy\n')

print('❌ TRAP 2: Forgetting groups=in_channels in depthwise conv')
print('   Wrong:  nn.Conv2d(128, 128, 3) → standard conv (expensive!)')
print('   Right:  nn.Conv2d(128, 128, 3, groups=128) → depthwise')
print('   Impact: 8× slower, defeats the purpose\n')

print('❌ TRAP 3: Scaling only one dimension (depth OR width OR resolution)')
print('   Wrong:  Double depth only → +1% accuracy for 2× FLOPs')
print('   Right:  EfficientNet compound scaling → +3–4% for 2× FLOPs')
print('   Why: Dimensions must scale together\n')

print('❌ TRAP 4: Deploying FP32 without quantization')
print('   Wrong:  MobileNetV2 FP32 → 14 MB, 35ms')
print('   Right:  MobileNetV2 INT8 → 3.5 MB, 12ms (-0.5% accuracy)')
print('   Impact: 4× model size, 3× latency overhead\n')

print('✅ Follow these rules and MobileNets deploy smoothly on edge!')